In [ ]:
import json
from pathlib import Path
from datetime import datetime
from funciones_auxiliares import extract_submissions, extract_comments_for_submissions, stream_zst_file

# --- 3 SUBREDDITS DANIELA ---
mis_subreddits = ["travel", "MealPrepSunday", "Fitness", "space", "investing", "LeagueOfLegends"] 

# Parámetros solicitados en el enunciado 
HILOS_POR_SUBREDDIT = 70
COMENTARIOS_POR_HILO_SUBMISSION = 150
COMENTARIOS_POR_HILO_COMMENT = 50

todas_submissions = []

# 1. Extraer los 70 hilos (submissions)
# Filtramos hilos que tengan al menos 50 comentarios para asegurar el corpus
todas_submissions = extract_submissions(
    "datos/RS_2025.zst", 
    mis_subreddits, 
    n_submissions=HILOS_POR_SUBREDDIT, 
    min_comments=COMENTARIOS_POR_HILO_SUBMISSION
)

# 2. Extraer todos los comentarios de golpe
extract_comments_for_submissions(
    "datos/RC_2025.zst", 
    todas_submissions, 
    num_comments=COMENTARIOS_POR_HILO_COMMENT
)

# 3 y 4. Separar la "superlista" y guardar en JSONs individuales
print("Generando archivos JSON individuales...")

for sub in mis_subreddits:
    # Filtramos la lista global para quedarnos solo con los hilos de ESTE subreddit
    submissions_del_sub = [s for s in todas_submissions if s.get('subreddit', '').lower() == sub.lower()]
    
    if not submissions_del_sub:
        print(f"⚠️ No se encontraron hilos suficientes para r/{sub}")
        continue

    # Estructurar el resultado
    resultado_final = {
        "subreddit": sub,
        "extraction_date": datetime.now().isoformat(),
        "num_submissions": len(submissions_del_sub),
        "total_comments": sum(len(s['comments']) for s in submissions_del_sub),
        "submissions": submissions_del_sub
    }

    # Guardar en JSON individual
    nombre_archivo = f"ejemplo_subreddit_{sub}.json"
    with open(nombre_archivo, 'w', encoding='utf-8') as f:
        json.dump(resultado_final, f, ensure_ascii=False, indent=2)

    print(f"✅ Archivo '{nombre_archivo}' generado con éxito.")


🔍 Buscando 70 submissions para 6 subreddits en una pasada...
  ✓ [r/fitness] 1/70 -> Daily Simple Questions Thread - January ...
  ✓ [r/fitness] 2/70 -> Rant Wednesday...
  ✓ [r/leagueoflegends] 1/70 -> Champions that are still fun even if you...
  ✓ [r/travel] 1/70 -> HELP on deciding first trip out of the c...
  ✓ [r/travel] 2/70 -> To those who have visited the Caribbean:...
  ✓ [r/leagueoflegends] 2/70 -> It's absurd that Gragas is allowed to ex...
  ✓ [r/leagueoflegends] 3/70 -> Los Ratones are confirmed to play in the...
⏳ Escaneadas 100,000 líneas... Estado actual: {'travel': 2, 'mealprepsunday': 0, 'fitness': 2, 'space': 0, 'investing': 0, 'leagueoflegends': 3}
  ✓ [r/investing] 1/70 -> Better to pay off mortgage at 4% or inve...
  ✓ [r/travel] 3/70 -> What city you've been to most surpassed ...
  ✓ [r/leagueoflegends] 4/70 -> Just lost to a 3-17 Sion...
  ✓ [r/leagueoflegends] 5/70 -> What is one champion you see complain th...
  ✓ [r/investing] 2/70 -> If everyone simply says

KeyboardInterrupt: 

In [ ]:
import json
from datetime import datetime

# Lista de los archivos que generaste anteriormente
archivos_json = ["ejemplo_subreddit_travel.json", "ejemplo_subreddit_MealPrepSunday.json", "ejemplo_subreddit_Fitness.json",
                  "ejemplo_subreddit_space.json", "ejemplo_subreddit_investing.json", "ejemplo_subreddit_LeagueOfLegends.json"]

for archivo in archivos_json:
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"\n{'='*50}")
        print(f"📅 ANÁLISIS TEMPORAL: r/{data['subreddit']}")
        print(f"{'='*50}")

        # Extraer fechas de creación de las submissions (convertidas de UTC a datetime)
        # created_utc viene en los datos originales del volcado [cite: 58]
        fechas = [datetime.fromtimestamp(s['created_utc']) for s in data['submissions']]
        
        if fechas:
            fechas_ordenadas = sorted(fechas)
            primera = fechas_ordenadas[0]
            ultima = fechas_ordenadas[-1]
            rango_dias = (ultima - primera).days

            print(f"🔹 Primera publicación: {primera.strftime('%Y-%m-%d %H:%M')}")
            print(f"🔹 Última publicación:  {ultima.strftime('%Y-%m-%d %H:%M')}")
            print(f"🔹 Amplitud temporal:    {rango_dias} días")

            # Contar cuántas hay por día para ver la densidad
            dias = [f.strftime('%Y-%m-%d') for f in fechas]
            conteo_dias = {dia: dias.count(dia) for dia in set(dias)}
            
            print("\n📊 Distribución por días (Primeros 5 días detectados):")
            for dia in sorted(conteo_dias.keys())[:5]:
                print(f"   - {dia}: {conteo_dias[dia]} hilos")
            
            if rango_dias < 1:
                print("\n⚠️ ALERTA: Todos los hilos son del mismo día. Considera saltar registros en la extracción.")
            else:
                print("\n✅ El corpus presenta variedad temporal.")
        else:
            print("❌ No se encontraron fechas en las submissions.")

    except FileNotFoundError:
        print(f"⚠️ No se encontró el archivo: {archivo}")


📅 ANÁLISIS TEMPORAL: r/travel
🔹 Primera publicación: 2025-01-01 03:49
🔹 Última publicación:  2025-01-10 13:29
🔹 Amplitud temporal:    9 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-01: 7 hilos
   - 2025-01-02: 5 hilos
   - 2025-01-03: 7 hilos
   - 2025-01-04: 9 hilos
   - 2025-01-05: 13 hilos

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/MealPrepSunday
🔹 Primera publicación: 2025-01-03 21:25
🔹 Última publicación:  2025-03-29 01:23
🔹 Amplitud temporal:    84 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-03: 1 hilos
   - 2025-01-04: 1 hilos
   - 2025-01-05: 2 hilos
   - 2025-01-06: 1 hilos
   - 2025-01-11: 1 hilos

✅ El corpus presenta variedad temporal.

📅 ANÁLISIS TEMPORAL: r/Fitness
🔹 Primera publicación: 2025-01-01 11:00
🔹 Última publicación:  2025-02-11 11:00
🔹 Amplitud temporal:    41 días

📊 Distribución por días (Primeros 5 días detectados):
   - 2025-01-01: 2 hilos
   - 2025-01-02: 1 hilos
   - 2025-01-03

In [ ]:
import json
import re


def analizar_calidad(texto):
    # Patrones para detectar URLs y Emails
    tiene_url = bool(re.search(r'https?://\S+|www\.\S+', texto))
    tiene_email = bool(re.search(r'\S+@\S+\.\S+', texto))
    longitud = len(texto.split()) # Contamos palabras
    return longitud, tiene_url, tiene_email

for archivo in archivos_json:
    with open(archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total_comentarios = 0
    cortos = 0 # Menos de 5 palabras
    solo_links = 0
    con_email = 0
    
    for submission in data['submissions']:
        for comment in submission.get('comments', []):
            total_comentarios += 1
            cuerpo = comment.get('body', '')
            
            n_palabras, has_url, has_email = analizar_calidad(cuerpo)
            
            if n_palabras < 5:
                cortos += 1
            if has_url and n_palabras < 3: # Muy corto y con URL suele ser solo spam/link
                solo_links += 1
            if has_email:
                con_email += 1

    print(f"\n{'='*50}")
    print(f"🔍 CALIDAD DEL TEXTO: r/{data['subreddit']}")
    print(f"{'='*50}")
    print(f"✅ Total analizados: {total_comentarios}")
    print(f"⚠️ Comentarios muy cortos (< 5 palabras): {cortos} ({cortos/total_comentarios*100:.1f}%)")
    print(f"🔗 Comentarios que son casi solo URLs: {solo_links}")
    print(f"📧 Comentarios con emails: {con_email}")
    
    if cortos / total_comentarios > 0.2:
        print("💡 Sugerencia: El corpus tiene mucho 'ruido' (mensajes cortos). Deberías filtrar en el siguiente paso.")


🔍 CALIDAD DEL TEXTO: r/travel
✅ Total analizados: 3500
⚠️ Comentarios muy cortos (< 5 palabras): 330 (9.4%)
🔗 Comentarios que son casi solo URLs: 1
📧 Comentarios con emails: 0

🔍 CALIDAD DEL TEXTO: r/MealPrepSunday
✅ Total analizados: 3500
⚠️ Comentarios muy cortos (< 5 palabras): 459 (13.1%)
🔗 Comentarios que son casi solo URLs: 10
📧 Comentarios con emails: 0

🔍 CALIDAD DEL TEXTO: r/Fitness
✅ Total analizados: 3500
⚠️ Comentarios muy cortos (< 5 palabras): 505 (14.4%)
🔗 Comentarios que son casi solo URLs: 12
📧 Comentarios con emails: 0

🔍 CALIDAD DEL TEXTO: r/space
✅ Total analizados: 3500
⚠️ Comentarios muy cortos (< 5 palabras): 478 (13.7%)
🔗 Comentarios que son casi solo URLs: 4
📧 Comentarios con emails: 1

🔍 CALIDAD DEL TEXTO: r/investing
✅ Total analizados: 3500
⚠️ Comentarios muy cortos (< 5 palabras): 383 (10.9%)
🔗 Comentarios que son casi solo URLs: 8
📧 Comentarios con emails: 0

🔍 CALIDAD DEL TEXTO: r/LeagueOfLegends
✅ Total analizados: 3500
⚠️ Comentarios muy cortos (< 5 pa